# 🚀 Notebook do Professor (Demo) — Aula 14: Spec-Driven Development e Encerramento do Semestre

**Disciplina:** Prompt Engineering and Artificial Intelligence  
**Instituição:** FIAP — Ciência da Computação · 2026  
**Professor:** Jorge Luiz Gomes  
**Aula 14/14 — Última aula · Sem entrega avaliada**  
**1h40min**  
**A fronteira atual do código assistido por IA**  
**Sem notebook — aula conceitual e demonstrativa**  

---

## Como usar este notebook

- Cada célula corresponde a um slide de código da aula (a ordem é a da apresentação).
- Rode ao vivo enquanto explica o slide correspondente.
- A última seção traz as soluções completas dos exercícios de fixação.

---

# 🔬 Código da aula — slide a slide

### Slide 16 — LangSmith — observabilidade para agentes em produção

In [ ]:
!pip install langchain-ollama langchain-core langchain-classic -q

from langchain_ollama import ChatOllama
from google.colab import userdata
import os

# Definir a API key via variável de ambiente (Colab Secrets)
os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

In [ ]:
import os
from google.colab import userdata

# Habilitar LangSmith — basta setar env vars
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = userdata.get("LANGSMITH_KEY")
os.environ["LANGCHAIN_PROJECT"] = "agente-ckp03"

# A partir daqui: QUALQUER invoke() e tracado
# automaticamente. Zero mudanca no codigo.
resultado = agente.invoke({...})
# → trace disponivel em smith.langchain.com
# → ver todos os passos, tools, decisoes

# Free tier: 5.000 traces/mes — suficiente
# para desenvolvimento e demos

### Slide 18 — Seus CKPs como portfólio — como apresentar no GitHub

```
# CKP03 — Agente Inteligente: [Dominio]

## Demo ao vivo
[![Open in Gradio](badge-url)](gradio-link)

## O que o agente faz
- Busca em documentos do dominio (RAG)
- Pesquisa na web (DuckDuckGo)
- Calcula metricas do dominio

## Stack
LangChain 0.3 | ChromaDB | Gradio | Ollama

## Como rodar no Colab
[![Open in Colab](colab-badge)](colab-link)

## Exemplos de uso
Q: "Quais os prazos do contrato?"
A: [screenshot da resposta]
```

---

## 🏋️ Exercícios Resolvidos — versão professor (executar no Colab)

As quatro soluções prontas dos exercícios de fixação do notebook do aluno — rode em sala, uma a uma.


### Exercício 1 — Spec.md da próxima melhoria

**O que a solução demonstra:** o template do Exercício 1 do aluno já PREENCHIDO com a spec de referência para o domínio do grupo — Requisito sem mencionar implementação, 3 critérios "se X, então Y" e um fora-do-escopo honesto.

**Pontos a destacar em sala:**
- Comparar lado a lado com o andaime do aluno: cada lacuna virou um campo objetivo — nenhum critério menciona LangChain, node ou retriever, só comportamento observável.
- Sinal de qualidade: cada critério responde com um "sim/não" objetivo; o que não responde ainda é opinião.
- A revisão por pares é o controle de qualidade da própria spec: caso de borda não coberto vira critério novo ou entra no "fora do escopo".


In [ ]:
# Solução — template de spec.md preenchido (Exercício 1) — modelo para o domínio do grupo
spec_md = """
# Spec — Citação de fonte e honestidade nas respostas

## Requisito
O agente deve responder perguntas sobre os documentos do domínio citando a fonte usada.
Quando a resposta vier de cálculo, ele deve mostrar a expressão avaliada.
Se nenhuma fonte sustentar a resposta, ele deve dizer explicitamente que não sabe.

## Critérios de aceite
1. Se a pergunta citar uma cláusula existente, então a resposta contém o trecho da fonte.
2. Se a pergunta for um cálculo, então a resposta traz a expressão e o resultado.
3. Se a pergunta não tem base nos documentos nem é cálculo, então a resposta contém "não encontrei" em vez de conteúdo inventado.

## Fora do escopo
- Busca na web (DuckDuckGo)
- Memória entre sessões
"""
print(spec_md)


### Exercício 2 — Trace do agente no LangSmith

**O que a solução demonstra:** as 4 lacunas do andaime do Exercício 2 preenchidas — nome da variável de tracing e valor, nome do secret, nome do projeto e o input de teste — com zero mudança no código do agente.

**Pontos a destacar em sala:**
- São só 3 variáveis de ambiente: `TRACING_V2=true` liga o tracing, a key autentica via `userdata.get` (nunca no código) e `LANGCHAIN_PROJECT` agrupa os traces no painel.
- No trace: árvore completa da execução — passos não instrumentados, comparação lado a lado e custo total; o passo mais lento costuma ser a chamada ao LLM no node de resposta.


In [ ]:
# Solução — LangSmith: observabilidade por variáveis de ambiente (Exercício 2)
import os
from google.colab import userdata

# Lacuna 1 — liga o tracing
os.environ["LANGCHAIN_TRACING_V2"] = "true"
# Lacuna 2 — a key via Colab Secrets
os.environ["LANGCHAIN_API_KEY"] = userdata.get("LANGSMITH_KEY")
# Lacuna 3 — nome do projeto que agrupa os traces
os.environ["LANGCHAIN_PROJECT"] = "agente-ckp03"

# Lacuna 4 — rode o agente normalmente: ZERO mudança no código
# resultado = router_chain.invoke({"input": "Qual o prazo de garantia no contrato?"})
# → trace completo em smith.langchain.com: nós na ordem, inputs/outputs,
#   tokens e latência de cada passo — compare execuções lado a lado.
# Free tier: 5.000 traces/mês — suficiente para o semestre.


### Exercício 3 — Prompt sozinho vs. spec

**O que a solução demonstra:** a tabela do andaime do Exercício 3 PREENCHIDA — as decisões implícitas no prompt único contra os critérios "se X, então Y" da spec, linha por linha.

**Pontos a destacar em sala:**
- No prompt único ficam implícitas: a fonte dos dados, o formato da resposta, o que fazer sem contexto e o limite de iteração — a ferramenta decide, e cada decisão virá como surpresa.
- As linhas em que o prompt aceita o comportamento em silêncio (input ambíguo e fora do domínio) são exatamente as que o critério força a ter um comportamento definido.
- Fechamento: o prompt descreve o que pedir à ferramenta de coding; a spec define o contrato contra o qual o resultado é aceito — sem contrato, "funcionou" é opinião.


In [ ]:
# Solução — comparação prompt único × spec preenchida (Exercício 3)
comparacao = """
| Decisão                    | Prompt único        | Spec (critério)                                   |
|----------------------------|---------------------|----------------------------------------------------|
| Sem fonte no documento     | implícito           | Se não há base nos documentos, então "não sabe".   |
| Resposta vinda de cálculo  | implícito           | Se for cálculo, então expressão + resultado.       |
| Input fora do domínio      | aceito em silêncio  | Se fora do domínio, então "não encontrei".         |
| Limite de iteração         | definido pela tool  | Se iterações > MAX, então para e reporta.          |
"""
print(comparacao)


### Exercício 4 — README do portfólio do CKP

**O que a solução demonstra:** o template do andaime do Exercício 4 PREENCHIDO como modelo — domínio, 3 bullets do que o agente faz, stack, Secrets e ordem de execução, 2 exemplos pergunta/resposta e a retrospectiva do semestre em 3 linhas.

**Pontos a destacar em sala:**
- O teste do README: um colega roda o projeto lendo apenas ele — se precisar de explicação de fora, falta algo no README.
- Na spec da próxima melhoria: o mesmo sinal de qualidade do Exercício 1 — critérios "se X, então Y" e fora-do-escopo honesto.
- Na retrospectiva: os 3 CKPs (chatbot, RAG, agente) formam o portfólio — numa conversa de seleção, o que importa é mostrar a decisão por trás de cada escolha.


In [ ]:
# Solução — README do portfólio preenchido (Exercício 4) — modelo para o domínio do grupo
readme_ckp = """
# CKP03 — Agente Inteligente: Contratos de prestação de serviços

## O que o agente faz
- Responde perguntas sobre os documentos do domínio (RAG + ChromaDB, com citação da fonte)
- Pesquisa complementar na web (DuckDuckGo) quando o documento não basta
- Calcula métricas do domínio (router → calculadora, com a expressão avaliada)

## Stack
LangChain 0.3 | LangGraph | ChromaDB | Gradio | Ollama (gpt-oss:120b)

## Como rodar no Colab
1. Secrets: OLLAMA_API_KEY (e LANGSMITH_KEY para o trace)
2. Ordem das células: setup primeiro, depois o router, depois o grafo

## Exemplos de uso
Q: "Quais os prazos previstos no contrato?"
A: [resposta com o trecho citado da cláusula 5]

## Retrospectiva do semestre (3 linhas)
Os 3 CKPs (chatbot, RAG, agente) formam o portfólio; o router da Aula 12 e o grafo
da Aula 13 entraram no CKP03. Próxima trilha: observabilidade com LangSmith em produção.
"""
print(readme_ckp)


## 📚 Referências da aula

- Ferramenta GitHub Spec Kit — github.com/github/spec-kit
- Blog Anthropic Engineering — Claude Code e desenvolvimento agentic. anthropic.com/engineering
- Ferramenta Kiro (AWS) — IDE spec-first. kiro.dev
- Blog LangChain Blog e Documentação — tutoriais oficiais sempre atualizados. blog.langchain.dev
- Curso DeepLearning.AI Short Courses — LangChain, LangGraph, RAG, Agentes. learn.deeplearning.ai
- Curso Hugging Face Learn — fine-tuning, PEFT, LoRA, datasets. huggingface.co/learn
- Livro Marco, E. — Agentic Coding with Claude Code. Packt Publishing, 2025. O mecanismo técnico do planning mode (/plan) por trás da separação spec-execução desta aula.
- Livro Avila, R. D. — Architecting AI Software Systems. Packt, 2025. Cap. 8 — Insights and Future Directions: ferramentas de IA não substituem o pensamento arquitetural de alto nível, base da trilha de carreira desta aula.

---

**→ Depois desta aula** — Não há próxima aula — o semestre está completo
  
Seus próximos passos: consolide os 3 CKPs como portfólio, escolha uma trilha de aprofundamento (observabilidade, multiagentes ou fine-tuning) e continue construindo. A jornada de Prompt & Context Engineering não termina aqui.

---

*Copyright © 2026 Prof. Jorge Luiz Gomes · FIAP · Todos os direitos reservados.*